In [1]:
import pandas as pd
import numpy as np

In [18]:
# Load database
df_database = pd.read_csv('../data/car_database.csv', sep=';')

In [3]:
df_database.columns

Index(['car', 'version', 'engine', 'transmission', 'horsepower', 'torque',
       'city_fuel_economy', 'highway_fuel_economy', 'payload_capacity',
       'ground_clearance', 'hydraulic_power_steering',
       'electric_power_steering', 'front_power_windows', 'rear_power_windows',
       'power_door_locks', 'power_side_mirrors', 'infotainment_system',
       'alloy_wheels', 'airbag', 'abs', 'stability_control',
       'traction_control', 'hill_start_assist', 'tire_pressure_sensor',
       'twilight_sensor ', 'steering_wheel_adjustment', 'air_conditioning',
       'driver_seat_height_adjustment', 'keyless', 'rear_parking_sensors',
       'front_parking_sensors', 'rear_view_camera', 'fog_lights',
       'led_headlights', 'cruise_control', 'leather_seats',
       'leather_steering_wheel', 'roof_rails', 'cost', 'warranty'],
      dtype='str')

In [19]:

# ==============================================================================
# 1. CRITERIA CONFIG
# ==============================================================================

criteria = {
    'cost': 'min', 
    'horsepower': 'max', 
    'transmission': 'max', 
    'city_fuel_economy': 'max', 
    'ground_clearance': 'max', 
    'rear_power_windows': 'max', 
    'power_side_mirrors': 'max', 
    'infotainment_system': 'max', 
    'rear_parking_sensors': 'max', 
    'fog_lights': 'max', 
    'roof_rails': 'max'
}

# ==============================================================================
# 2. Hierarchy Definition 
# ==============================================================================

# --- Tiers ---
tier_0_essential    = ['cost']
tier_1_must_have   = ['infotainment_system', 'rear_parking_sensors', 'fog_lights']
tier_2_very_nice_to_have = ['horsepower', 'ground_clearance',  'roof_rails']
tier_3_nice_to_have   = ['transmission', 'rear_power_windows', 'power_side_mirrors', 'city_fuel_economy']

# --- Weight Neighbours ---
# Saaty Scale Reference: 1=Equal, 3=Moderate, 5=Strong, 7=Very Strong, 9=Extreme

# ESSENTIAL is 2x more important than MUST = Saaty 2 (Equal to Moderate)
STEP_ESSENTIAL_MUST   = 2.0

# MUST is 2x more important than VERYNICE  = Saaty 2 (Equal to Moderate)
# Cumulative Effect: ESSENTIAL vs. VERYNICE = Saaty 4 (Moderate to Strong).
STEP_MUST_VERYNICE = 2.0

# VERYNICE is 2x more important than NICE = Saaty 2 (Equal to Moderate)
# Cumulative Effect: ESSENTIAL vs. NICE = Saaty 8 (Very Strong to Extreme).
STEP_VERYNICE_NICE = 2.0 

# --- Automatic Crossing ---

# Function to cross two lists with a given weight
def apply_relative_importance(preferences_dict, strong_list, weak_list, weight):
    for strong in strong_list:
        for weak in weak_list:
            preferences_dict[(strong, weak)] = weight

# Dict of Preferences
preferences = {}

# 1. Neighbours Crossing (Directly Connected)
apply_relative_importance(preferences, tier_0_essential,  tier_1_must_have,   STEP_ESSENTIAL_MUST)
apply_relative_importance(preferences, tier_1_must_have, tier_2_very_nice_to_have, STEP_MUST_VERYNICE)
apply_relative_importance(preferences, tier_2_very_nice_to_have, tier_3_nice_to_have, STEP_VERYNICE_NICE)

# 2. Neighbours Crossing with two steps of distance
# ESSENTIAL -> VERYNICE (2 * 2 = 4)
apply_relative_importance(preferences, tier_0_essential, tier_2_very_nice_to_have, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE)
# MUST -> NICE (2 * 2 = 4)
apply_relative_importance(preferences, tier_1_must_have, tier_3_nice_to_have, STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)

# 3. Neighbours Crossing with three steps of distance
# ESSENTIAL -> NICE (2 * 2 * 2 = 8)
apply_relative_importance(preferences, tier_0_essential, tier_3_nice_to_have, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)

# ==============================================================================
# 3. AHP Engine
# ==============================================================================

def exec_ahp(df, config_crit, rules_pref):
    
    cols = list(criteria.keys())
    df_m = df[cols].copy()

    # 2. Matrix Building
    
    n = len(cols)
    matrix = np.ones((n, n)) 
    
    idx = {name: i for i, name in enumerate(cols)}
    for (a, b), v in rules_pref.items():
        if a in idx and b in idx:
            matrix[idx[a], idx[b]] = v
            matrix[idx[b], idx[a]] = 1/v
            
    # 3. Wheight

    col_sums = matrix.sum(axis=0)
    pesos = (matrix / col_sums).mean(axis=1)
    
    # 4. Consistency 

    lambda_max = (matrix.dot(pesos) / pesos).mean()
    ci = (lambda_max - n) / (n - 1)
    
    # RI Table Reference (Saaty, 1980)
    ri_dict = {1:0, 2:0, 3:0.58, 4:0.9, 5:1.12, 6:1.24, 7:1.32, 8:1.41, 9:1.45, 10:1.49, 11:1.51, 12:1.48}
    ri = ri_dict.get(n, 1.51)
    
    cr = ci / ri if ri > 0 else 0
    
    # 5. Score
    df_norm = pd.DataFrame(index=df_m.index)
    for c, obj in config_crit.items():
        v = df_m[c]
        if v.sum() == 0: df_norm[c] = 0
        elif obj == 'min': df_norm[c] = (1/v)/(1/v).sum()
        else: df_norm[c] = v/v.sum()
        
    df_m['Score_Final'] = df_norm.dot(pesos)
    
    # Show Result
    print(f"--- AHP Report (n={n}) ---")
    print(f"Consistency (CR): {cr:.5f} {'(Great)' if cr < 0.1 else '(Attention)'}")
    
    return df_m.sort_values('Score_Final', ascending=False)

# ==============================================================================
# 4. Execution
# ==============================================================================

# 1. Data Cleanup
if 'transmission' in df_database.columns:
    df_database['transmission'] = df_database['transmission'].apply(lambda x: 1 if str(x).lower().strip() == 'automatic' else 0)
if 'ground_clearance' in df_database.columns:
    df_database['ground_clearance'] = df_database['ground_clearance'].apply(lambda x: 2 if str(x).lower().strip() == 'high' else 1 if str(x).lower().strip() == 'medium' else 0)
for col in criteria.keys():
    if df_database[col].dtype == bool: df_database[col] = df_database[col].astype(int)

df_database['car_version'] = df_database['car'] + ' - ' + df_database['version']
df_database.set_index('car_version', inplace=True)

ranking = exec_ahp(df_database, criteria, preferences)

# Show most relevant columns + Score
cols_view = ['cost', 'infotainment_system', 'rear_parking_sensors', 'fog_lights', 'horsepower', 'Score_Final']
ranking[cols_view].head(10).round(4)

--- AHP Report (n=11) ---
Consistency (CR): 0.00000 (Great)


,cost,infotainment_system,rear_parking_sensors,fog_lights,horsepower,Score_Final
car_version,,,,,,
Citroën C3 2026 - 1.0 TURBO 200 FLEX YOU CVT,110280.00,1,1,True,125,0.057221
Fiat Argo 2026 - 1.3 FIREFLY FLEX TREKKING CVT + Trekking TOP,113880.00,1,1,True,101,0.056527
Fiat Argo 2026 - 1.3 FIREFLY FLEX TREKKING MANUAL,102790.00,1,1,True,101,0.054764
Fiat Argo 2026 - 1.3 FIREFLY FLEX TREKKING MANUAL + Trekking TOP,105880.00,1,1,True,101,0.054509
Fiat Argo 2026 - 1.3 FIREFLY FLEX TREKKING CVT,110790.00,1,1,True,101,0.054133
Citroën C3 2026 - 1.0 FIREFLY FLEX XTR MANUAL,91290.00,1,0,True,71,0.046936
Peugeot 208 2026 - 1.0 TURBO 200 FLEX ACTIVE CVT + Sensor,111470.42,1,1,True,125,0.045479
Peugeot 208 2026 - 1.0 FIREFLY FLEX STYLE MANUAL,95990.00,1,1,True,71,0.043091
Fiat Argo 2026 - 1.0 FIREFLY FLEX DRIVE MANUAL + PACK DRIVE TOP,99090.00,1,1,True,72,0.042821
